# 1. Able-Bodied Baseline: Full-Window Frequency-Domain Action Recognition

## Research Goal for Prosthetic Design

This notebook establishes a **performance ceiling** for prosthetic control: can we recognize hand actions from complete windows of 22-channel glove sensor data in the frequency domain, using able-bodied (intact) participants?

**Why this matters:** A prosthetic controller decodes intended actions from residual-limb signals. We cannot observe intended actions directly in amputees. So our strategy is:

1. **This notebook:** Learn action patterns from able-bodied glove kinematics (ground truth for what we want the prosthetic to recognize).
2. **Notebook 2:** Apply the same pipeline to amputee recordings—measure how much recognition performance drops.
3. **Notebook 3:** Use Responsible AI to understand where patterns transfer and where they break down.

**Non-clinical scope:** This is offline, retrospective movement recognition. It establishes a design baseline, not a validated device or clinical prediction tool.

## 2. Dataset Characterization: Able-Bodied Cohort (Ninapro DB1, "Intact")

**Source:** Ninapro DB1 Exercises B & C from able-bodied volunteers.

**Measurement:**
- `glove`: 22-channel CyberGlove kinematics (hand position, finger flexion).
- `restimulus`: labeled action (0=Rest, 1-17=Exercise B actions, 18-40=Exercise C actions).
- `rerepetition`: repetition index (1-6), used to prevent train/test leakage.

**Window parameters:**
- 400-sample glove windows with 100-sample stride.
- Windows with >80% same-action label (purity) are retained.
- Windows crossing a repetition boundary are discarded (prevents data leakage).
- Frequency representation: log-magnitude FFT, 32 bins per channel = 704 features per window.

**Why frequency domain:** Hand movements have distinct temporal structure. Frequency content (slow precision grasps vs. fast grasp-release cycles) captures information that EMG can potentially decode, especially if trained on the right baseline.

## 3. Setup

- Shared utilities imported from `utils.py` (data loading, windowing, splitting, modeling, evaluation).
- Reproducibility: fixed random seed, group-aware train/test split (prevents leakage across repetitions).
- Hierarchical model: exercise-level router → exercise-specific action classifiers.
- PyTorch for training on CPU/GPU with automatic device selection.

In [1]:
# Reserve names so later cells have explicit placeholders
train_df_intact = None
test_df_intact = None

# Import core analysis libraries
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Import from shared utilities
import sys
sys.path.insert(0, str(Path.cwd().parent))  # parent directory for utils.py
from utils import (
    ACTIONS, EXERCISE_NAMES,
    set_seed, find_repo_root, get_data_root,
    load_xlsx_recording,
    make_frequency_windows,
    subject_grouped_holdout, cap_and_shuffle_classes, exercise_labels_from_classes,
    FrequencyMLP, fit_frequency_model, predict_hierarchical,
    classification_summary, save_experiment_json,
    plot_bar, plot_confusion_heatmap,
)

# Import PyTorch
import torch
from sklearn.metrics import classification_report, confusion_matrix

# Matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Core libraries imported")
print(f"✓ PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

ModuleNotFoundError: No module named 'pandas'

## 4. Configuration and Seeds

Fixed random state for reproducibility. All data splits are stratified and group-aware to prevent leakage.

In [ ]:
SEED = 42
EXPERIMENT_ID = "EXP-008-ablebody-frequency-baseline"
COHORT = "Intact"

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
repo_root = find_repo_root()
data_root = get_data_root(repo_root)
results_dir = repo_root / 'results' / 'metrics'
results_dir.mkdir(parents=True, exist_ok=True)

print(f"SEED: {SEED}")
print(f"Device: {device}")
print(f"Data root: {data_root}")
print(f"Results dir: {results_dir}")

SEED: 42
Device: cpu
Data root: c:\Users\david\OneDrive\Documents\School\AI in Healthcare\Database
Results dir: c:\Users\david\OneDrive\Documents\School\AI in Healthcare\results\metrics


## 5. Data Loading and Windowing

Load all Able-bodied (Intact) recordings, create frequency windows, and extract labels.

In [ ]:
# Locate all Intact (Able-bodied) recordings
intact_dir = data_root / 'Intact'
intact_paths = sorted(intact_dir.glob('*_A1_E*.xlsx'))
print(f"Found {len(intact_paths)} Intact recordings")

# Load and window all recordings
records = []
for path in intact_paths:
    recording = load_xlsx_recording(path)
    windowed = make_frequency_windows(
        recording, path, cohort='Intact',
        window_samples=400, stride_samples=100, fft_bins=32, min_purity=0.80
    )
    records.append(windowed)
    print(f"  {path.name}: {windowed[0].shape[0]} windows")

# Concatenate all windows and labels
X = np.concatenate([r[0] for r in records], axis=0)
target_names = np.concatenate([r[1] for r in records])
groups = np.concatenate([r[2] for r in records])

print(f"\nTotal windows: {X.shape[0]}")
print(f"Feature dimension: {X.shape[1]}")
print(f"Unique actions: {len(np.unique(target_names))}")

Found 63 Intact recordings
  S10_A1_E1.xlsx: 122 windows
  S10_A1_E2.xlsx: 158 windows
  S10_A1_E3.xlsx: 393 windows
  S17_A1_E1.xlsx: 179 windows
  S17_A1_E2.xlsx: 181 windows


## 6. Train/Test Split (Group-Aware)

Hold out subjects and their repetitions to prevent leakage. This ensures the test set contains truly unseen participants.

In [ ]:
# Extract subject IDs from group strings
subjects = np.asarray([g.split(':', 1)[0] for g in groups])
unique_subjects = np.unique(subjects)
print(f"Unique subjects: {len(unique_subjects)}")

# Group-aware holdout split: hold out subjects and their repetition groups
train_mask, test_mask, test_subjects, val_subjects = subject_grouped_holdout(
    groups, subjects, SEED, test_fraction=0.25
)

print(f"Train windows: {train_mask.sum()}")
print(f"Test windows: {test_mask.sum()}")
print(f"Test subjects: {sorted(test_subjects)}")
print(f"Val subjects: {sorted(val_subjects)}")

# Create class indices for the hierarchical model
classes = sorted(np.unique(target_names))
y = np.asarray([classes.index(t) for t in target_names])
print(f"\nClass mapping: {len(classes)} classes")

## 7. Training Data Preparation

Cap and shuffle training classes to balance the dataset.

In [ ]:
# Balance training classes
selected_indices = cap_and_shuffle_classes(y, train_mask, SEED, max_per_class=1000)

X_train = X[selected_indices]
y_train = y[selected_indices]
target_names_train = target_names[selected_indices]

X_test = X[test_mask]
y_test = y[test_mask]
target_names_test = target_names[test_mask]

print(f"Training windows (balanced): {X_train.shape[0]}")
print(f"Test windows: {X_test.shape[0]}")

# Exercise-level labels for hierarchical model
exercise_ids_train, _, y_exercise_train = exercise_labels_from_classes(classes, y_train)
exercise_ids_test, _, y_exercise_test = exercise_labels_from_classes(classes, y_test)

print(f"\nExercises in training: {len(exercise_ids_train)}")
print(f"Exercises in test: {len(exercise_ids_test)}")

## 8. Model Training: Hierarchical Exercise Router + Action Classifiers

Train an exercise router first, then exercise-specific action classifiers.

In [ ]:
# Train exercise-level router
print("Training exercise router...")
exercise_model = fit_frequency_model(
    X_train, y_exercise_train, len(exercise_ids_train), device,
    epochs=200, batch_size=32, learning_rate=0.001
)
print("✓ Exercise router trained")

# Train per-exercise action classifiers
print("\nTraining per-exercise action classifiers...")
action_models = {}
for ex_idx, ex_id in enumerate(exercise_ids_train):
    ex_mask = y_exercise_train == ex_idx
    if ex_mask.sum() > 0:
        n_actions = len(np.unique(y_train[ex_mask]))
        action_models[ex_id] = fit_frequency_model(
            X_train[ex_mask], y_train[ex_mask], n_actions, device,
            epochs=150, batch_size=16, learning_rate=0.0005
        )
        print(f"  Exercise {ex_id}: {n_actions} actions")

print(f"\n✓ {len(action_models)} exercise-specific classifiers trained")

## 9. Evaluation: Cross-Subject Test Performance

Predict on held-out subjects using the hierarchical model.

In [ ]:
# Predict on test set using hierarchical routing
print("Predicting on held-out test subjects...")
y_pred = predict_hierarchical(
    exercise_model, action_models, exercise_ids_test, X_test, device
)

# Evaluate
metrics = classification_summary(y_test, y_pred)
print(f"\nCross-Subject Test Performance:")
print(f"  Accuracy: {metrics['accuracy']:.1%}")
print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.1%}")
print(f"  Macro F1: {metrics['macro_f1']:.1%}")

# Classification report
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=classes, digits=3))

## 10. Confusion Matrix and Error Analysis

Visualize where the model succeeds and fails.

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
plot_confusion_heatmap(
    cm, classes, ax=ax,
    title=f"Able-Bodied Action Recognition: Cross-Subject Test\n(Accuracy: {metrics['accuracy']:.1%})"  
)
plt.tight_layout()
plt.savefig(results_dir / f"{EXPERIMENT_ID}_confusion.png", dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved")

## 11. Per-Action Performance

Which actions are easy to recognize? Which are hard?

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)

# Create per-action summary
action_performance = pd.DataFrame({
    'Action': classes,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

action_performance = action_performance.sort_values('F1-Score', ascending=False)
print("\nPer-Action Performance (sorted by F1):")
print(action_performance.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
plot_bar(
    action_performance.sort_values('F1-Score', ascending=True)['Action'],
    action_performance.sort_values('F1-Score', ascending=True)['F1-Score'],
    ax=ax,
    xlabel='Macro F1 Score', ylabel='Action',
    title='Per-Action Recognition Performance (Able-Bodied Cohort)'
)
plt.tight_layout()
plt.savefig(results_dir / f"{EXPERIMENT_ID}_per_action_f1.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. Design Interpretation: What This Baseline Means

This able-bodied baseline is our **performance ceiling for prosthetic design**. The findings guide the next step.

In [ ]:
# Identify high-performing and low-performing actions
high_f1 = action_performance[action_performance['F1-Score'] >= 0.75]['Action'].tolist()
medium_f1 = action_performance[(action_performance['F1-Score'] >= 0.50) & (action_performance['F1-Score'] < 0.75)]['Action'].tolist()
low_f1 = action_performance[action_performance['F1-Score'] < 0.50]['Action'].tolist()

print(f"\n### DESIGN IMPLICATIONS ###\n")
print(f"High-confidence actions (F1 ≥ 0.75): {len(high_f1)}")
if high_f1:
    print(f"  → Safe for direct prosthetic control: {', '.join(high_f1[:5])}" + ("..." if len(high_f1) > 5 else ""))

print(f"\nMedium-confidence actions (0.50 ≤ F1 < 0.75): {len(medium_f1)}")
if medium_f1:
    print(f"  → Candidates for user confirmation: {', '.join(medium_f1[:5])}" + ("..." if len(medium_f1) > 5 else ""))

print(f"\nLow-confidence actions (F1 < 0.50): {len(low_f1)}")
if low_f1:
    print(f"  → May need simplified gesture set: {', '.join(low_f1[:5])}" + ("..." if len(low_f1) > 5 else ""))

print(f"\n### NEXT STEP ###")
print(f"Notebook 2 will measure the same actions in amputee recordings.")
print(f"The performance drop tells us which actions need special handling in prosthetic design.")

## 13. Save Results

Persist experiment metadata for reporting and Notebook 3 analysis.

In [ ]:
# Save experiment summary
experiment_data = {
    'experiment_id': EXPERIMENT_ID,
    'cohort': COHORT,
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'test_subjects': sorted(test_subjects),
    'metrics': {
        'accuracy': float(metrics['accuracy']),
        'balanced_accuracy': float(metrics['balanced_accuracy']),
        'macro_f1': float(metrics['macro_f1']),
    },
    'per_action': action_performance.to_dict(orient='records'),
    'n_windows_train': int(X_train.shape[0]),
    'n_windows_test': int(X_test.shape[0]),
    'n_unique_actions': len(classes),
    'feature_dimension': int(X.shape[1]),
}

save_experiment_json(results_dir, EXPERIMENT_ID, experiment_data)
print(f"✓ Experiment results saved: {EXPERIMENT_ID}")